# 03_heart_cluster_annotation

Selected source cells for review and reconstruction.

Source: `Heart_combined_samples/Heart_Combination_2-3(Cluster naming).ipynb`

Requires BioMart-renamed Seurat object; final Npr3 cutoff retained after clearing initial cluster8 labels.

Original execution counts and plots were removed. Do not run in the source-data folder: original output filenames and Docker paths are retained as provenance. Use the common scripts for new runs. These excerpts have not been executed as a fresh notebook.


Source cell 1 (one-based); original execution count 1.

In [ ]:
library(Seurat)

Source cell 2 (one-based); original execution count 2.

In [ ]:
obj <- readRDS("heart_mergeobj_preProject_full_sketchHarmony_ENSMUSG16kept.rds")

obj
Assays(obj)
Reductions(obj)

# 重要：どのreductionがあるか
names(obj@reductions)

# 重要：ENSMUSGが16個残っているか（RNAのrownamesで確認）
sum(grepl("^ENSMUSG", rownames(obj[["RNA"]])))
head(rownames(obj[["RNA"]])[grepl("^ENSMUSG", rownames(obj[["RNA"]]))])

# sketch / full の状態確認（存在すれば）
DefaultAssay(obj)
lapply(Assays(obj), function(a) dim(obj[[a]]))


Source cell 4 (one-based); original execution count 3.

In [ ]:
DefaultAssay(obj) <- "RNA"
Idents(obj) <- obj$cluster_full

# ★ここが必須
obj <- JoinLayers(obj)   # これでRNAのdata/countsが統合される

markers <- FindAllMarkers(
  obj,
  only.pos = TRUE,
  min.pct = 0.25,
  logfc.threshold = 0.25
)

# 念のため列名確認
colnames(markers)[1:10]


Source cell 8 (one-based); original execution count 10.

In [ ]:
obj$cluster_naming <- NA_character_
cl <- as.character(obj$cluster_full)

obj$cluster_naming[cl == "0"]  <- "Atrial cardiac muscle"
obj$cluster_naming[cl == "16"] <- "Platelet"
obj$cluster_naming[cl == "10"] <- "Red blood cell"
obj$cluster_naming[cl == "23"] <- "Airway epithelium"
obj$cluster_naming[cl == "17"] <- "Airway structural tissue"
obj$cluster_naming[cl == "26"] <- "Cardiac neuron"
obj$cluster_naming[cl == "8"]  <- "Endocardium"
obj$cluster_naming[cl == "12"] <- "Epicardium"
obj$cluster_naming[cl == "5"]  <- "Adipose tissue"

table(obj$cluster_naming, useNA = "ifany")


Source cell 12 (one-based); original execution count 15.

In [ ]:
# Endocardium を一旦 NA に戻す
obj$cluster_naming[obj$cluster_naming == "Endocardium"] <- NA_character_

# 確認
table(obj$cluster_naming, useNA = "ifany")


Source cell 22 (one-based); original execution count 27.

In [ ]:
npr3 <- FetchData(obj, "Npr3")[,1]
in8  <- as.character(obj$cluster_full) == "8"

# cutoff 0.01
obj$cluster_naming[in8 & (npr3 >= 0.01)] <- "Endocardium"


Source cell 25 (one-based); original execution count 30.

In [ ]:
# meta.data から endo_by_npr3 列を削除
obj$endo_by_npr3 <- NULL

# 確認（列名に endo_by_npr3 が無いこと）
"endo_by_npr3" %in% colnames(obj@meta.data)


Source cell 27 (one-based); original execution count 32.

In [ ]:
saveRDS(
  obj,
  file = "heart_mergeobj_preProject_full_sketchHarmony_ENSMUSG16kept_clusterNaming.rds"
)